In [ ]:
import warnings
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
import pygeohash as pgh
from sklearn.metrics import r2_score

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

RANDOM_STATE = 42
DATA_DIR = Path('.')
TRAIN_PATH = DATA_DIR / 'train.csv'
TEST_PATH = DATA_DIR / 'test.csv'
SUBMISSION_PATH = DATA_DIR / 'submission.csv'

LAG_FEATURES = [1, 2, 4, 8, 12, 24, 96]
ROLLING_WINDOWS = [4, 8, 24]
DYNAMIC_FEATURE_COLUMNS = [
    *[f'demand_lag_{lag}' for lag in LAG_FEATURES],
    *[f'demand_roll_mean_{window}' for window in ROLLING_WINDOWS],
    *[f'demand_roll_std_{window}' for window in ROLLING_WINDOWS],
]


def parse_timestamp_to_minutes(timestamp_series: pd.Series) -> pd.Series:
    '''Convert HH:MM strings into minutes past midnight.'''
    parts = timestamp_series.astype(str).str.split(':', n=1, expand=True)
    hours = parts[0].astype(int)
    minutes = parts[1].astype(int)
    return hours * 60 + minutes



def decode_single_geohash(geohash_value: str) -> tuple[float, float]:
    '''Decode one geohash string into latitude and longitude.'''
    if pd.isna(geohash_value):
        return np.nan, np.nan

    try:
        decoded = pgh.decode_exactly(str(geohash_value))
        if hasattr(decoded, 'latitude') and hasattr(decoded, 'longitude'):
            return float(decoded.latitude), float(decoded.longitude)
        return float(decoded[0]), float(decoded[1])
    except Exception:
        try:
            decoded = pgh.decode(str(geohash_value))
            if hasattr(decoded, 'latitude') and hasattr(decoded, 'longitude'):
                return float(decoded.latitude), float(decoded.longitude)
            return float(decoded[0]), float(decoded[1])
        except Exception:
            return np.nan, np.nan



def add_geohash_coordinates(frame: pd.DataFrame) -> pd.DataFrame:
    '''Add latitude and longitude columns derived from geohash values.'''
    geohash_series = frame['geohash'].astype('string')
    unique_geohashes = geohash_series.dropna().unique()
    geohash_lookup = {value: decode_single_geohash(value) for value in unique_geohashes}

    coordinates = geohash_series.map(geohash_lookup)
    frame['latitude'] = [pair[0] if isinstance(pair, tuple) else np.nan for pair in coordinates]
    frame['longitude'] = [pair[1] if isinstance(pair, tuple) else np.nan for pair in coordinates]
    return frame



def add_lag_and_rolling_features(frame: pd.DataFrame) -> pd.DataFrame:
    '''Add autoregressive lags and rolling demand statistics by geohash.'''
    grouped = frame.groupby('geohash', sort=False)

    for lag in LAG_FEATURES:
        frame[f'demand_lag_{lag}'] = grouped['demand'].shift(lag)

    lagged_demand = grouped['demand'].shift(1)
    grouped_lagged = lagged_demand.groupby(frame['geohash'], sort=False)

    for window in ROLLING_WINDOWS:
        frame[f'demand_roll_mean_{window}'] = grouped_lagged.transform(
            lambda series: series.rolling(window=window, min_periods=1).mean()
        )
        frame[f'demand_roll_std_{window}'] = grouped_lagged.transform(
            lambda series: series.rolling(window=window, min_periods=1).std()
        )

    geohash_demand_mean = frame.groupby('geohash')['demand'].transform('mean')
    frame[DYNAMIC_FEATURE_COLUMNS] = frame[DYNAMIC_FEATURE_COLUMNS].fillna(geohash_demand_mean)
    return frame



def compute_dynamic_features(history_values: list[float]) -> dict[str, float]:
    '''Build lag and rolling features from the available demand history.'''
    feature_values: dict[str, float] = {}

    for lag in LAG_FEATURES:
        feature_values[f'demand_lag_{lag}'] = history_values[-lag] if len(history_values) >= lag else np.nan

    for window in ROLLING_WINDOWS:
        window_values = history_values[-window:]
        if window_values:
            feature_values[f'demand_roll_mean_{window}'] = float(np.mean(window_values))
            feature_values[f'demand_roll_std_{window}'] = float(np.std(window_values, ddof=1)) if len(window_values) > 1 else 0.0
        else:
            feature_values[f'demand_roll_mean_{window}'] = np.nan
            feature_values[f'demand_roll_std_{window}'] = np.nan

    return feature_values



def recursive_predict(
    model: lgb.LGBMRegressor,
    history_df: pd.DataFrame,
    target_df: pd.DataFrame,
    feature_columns: list[str],
    categorical_columns: list[str],
    category_levels: dict[str, pd.Index],
) -> pd.Series:
    '''Predict rows sequentially while updating lag features with prior predictions.'''
    if target_df.empty:
        return pd.Series(dtype=float)

    static_columns = [column for column in feature_columns if column not in DYNAMIC_FEATURE_COLUMNS]
    predictions: list[float] = []
    prediction_index: list[int] = []

    history_by_geohash: dict[str, list[float]] = {
        str(geohash_value): group.sort_values(['day', 'time_mins'])['demand'].astype(float).tolist()
        for geohash_value, group in history_df.groupby('geohash', sort=False)
    }

    sorted_target_df = target_df.sort_values(['geohash', 'day', 'time_mins'], kind='mergesort')
    for geohash_value, geohash_group in sorted_target_df.groupby('geohash', sort=False):
        geohash_key = str(geohash_value)
        history_values = history_by_geohash.get(geohash_key)
        if history_values is None:
            history_values = []
            history_by_geohash[geohash_key] = history_values

        for row in geohash_group.itertuples(index=False):
            feature_row = {column: getattr(row, column) for column in static_columns}
            feature_row.update(compute_dynamic_features(history_values))
            feature_frame = pd.DataFrame([feature_row], columns=feature_columns)

            for column in categorical_columns:
                feature_frame[column] = pd.Categorical(feature_frame[column], categories=category_levels[column])

            prediction_values = np.asarray(model.predict(feature_frame), dtype=float)
            prediction = float(prediction_values[0])
            predictions.append(prediction)
            prediction_index.append(int(getattr(row, 'Index')))
            history_values.append(prediction)

    return pd.Series(predictions, index=prediction_index)


In [2]:
train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)

train_raw['is_test'] = False
test_raw['is_test'] = True
test_raw['demand'] = np.nan

combined_df = pd.concat([train_raw, test_raw], ignore_index=True, sort=False)
combined_df['time_mins'] = parse_timestamp_to_minutes(combined_df['timestamp']).astype(int)
combined_df['time_sin'] = np.sin(2 * np.pi * combined_df['time_mins'] / 1440.0)
combined_df['time_cos'] = np.cos(2 * np.pi * combined_df['time_mins'] / 1440.0)
combined_df = add_geohash_coordinates(combined_df)

combined_df.head()

,Index,geohash,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather,is_test,time_mins,time_sin,time_cos,latitude,longitude
0,0,qp02z1,48,0:0,0.048804,NaN,1,Not Allowed,No,NaN,NaN,False,0,0.0,1.0,-5.484924,90.664673
1,1,qp02zt,48,0:0,0.118507,Residential,3,Allowed,Yes,31.104565,Sunny,False,0,0.0,1.0,-5.462952,90.686646
2,2,qp08bj,48,0:0,0.027132,Residential,1,Not Allowed,No,25.919267,Sunny,False,0,0.0,1.0,-5.462952,90.708618
3,3,qp08gt,48,0:0,0.003272,Residential,1,Not Allowed,No,NaN,Rainy,False,0,0.0,1.0,-5.462952,90.862427
4,4,qp02zq,48,0:0,0.010819,Residential,1,Not Allowed,No,10.803667,Rainy,False,0,0.0,1.0,-5.457458,90.675659


In [6]:
combined_df = combined_df.sort_values(['geohash', 'day', 'time_mins'], kind='mergesort').reset_index(drop=True)

combined_df['Temperature'] = combined_df.groupby('geohash')['Temperature'].transform(lambda series: series.ffill().bfill())
combined_df['Weather'] = combined_df.groupby('geohash')['Weather'].transform(lambda series: series.ffill().bfill())
combined_df['RoadType'] = combined_df['RoadType'].fillna('Unknown')

combined_df = add_lag_and_rolling_features(combined_df)

categorical_columns = ['geohash', 'RoadType', 'Weather', 'LargeVehicles', 'Landmarks']
for column_name in categorical_columns:
    combined_df[column_name] = combined_df[column_name].astype('category')

combined_df.dtypes

Index                     int64
geohash                category
day                       int64
timestamp                object
demand                  float64
RoadType               category
NumberofLanes             int64
LargeVehicles          category
Landmarks              category
Temperature             float64
Weather                category
is_test                    bool
time_mins                 int32
time_sin                float64
time_cos                float64
latitude                float64
longitude               float64
demand_lag_1            float64
demand_lag_2            float64
demand_lag_4            float64
demand_lag_8            float64
demand_lag_12           float64
demand_lag_24           float64
demand_lag_96           float64
demand_roll_mean_4      float64
demand_roll_std_4       float64
demand_roll_mean_8      float64
demand_roll_std_8       float64
demand_roll_mean_24     float64
demand_roll_std_24      float64
dtype: object

In [8]:
train_df = combined_df.loc[~combined_df['is_test']].copy()
test_df = combined_df.loc[combined_df['is_test']].copy()

feature_columns = [
    'geohash',
    'day',
    'time_mins',
    'time_sin',
    'time_cos',
    'latitude',
    'longitude',
    'RoadType',
    'NumberofLanes',
    'LargeVehicles',
    'Landmarks',
    'Temperature',
    'Weather',
    'demand_lag_1',
    'demand_lag_2',
    'demand_lag_4',
    'demand_lag_8',
    'demand_lag_12',
    'demand_lag_24',
    'demand_lag_96',
    'demand_roll_mean_4',
    'demand_roll_mean_8',
    'demand_roll_mean_24',
    'demand_roll_std_4',
    'demand_roll_std_8',
    'demand_roll_std_24',
]

train_mask = (train_df['day'] == 48)
valid_mask = (train_df['day'] == 49) & (train_df['time_mins'] <= 120)

X_train = train_df.loc[train_mask, feature_columns].copy()
y_train = train_df.loc[train_mask, 'demand'].astype(float)
X_valid = train_df.loc[valid_mask, feature_columns].copy()
y_valid = train_df.loc[valid_mask, 'demand'].astype(float)


def build_regressor(device_type: str, n_estimators: int) -> lgb.LGBMRegressor:
    '''Create a LightGBM regressor with the requested accelerator mode.'''
    return lgb.LGBMRegressor(
        objective='regression',
        random_state=RANDOM_STATE,
        n_estimators=n_estimators,
        learning_rate=0.05,
        device_type=device_type,
        n_jobs=-1,
        verbosity=-1,
        max_depth=-1,
        num_leaves=255,
        min_child_samples=20,
        subsample=0.9,
        subsample_freq=1,
        colsample_bytree=0.9,
        reg_alpha=0.1,
        reg_lambda=0.2,
    )


try:
    model = build_regressor('cuda', 1500)
    model.fit(
        X_train,
        y_train,
        eval_set=[(X_valid, y_valid)],
        eval_metric='rmse',
        categorical_feature=categorical_columns,
        callbacks=[lgb.early_stopping(50, verbose=True)],
    )
    trained_device = 'cuda'
except lgb.basic.LightGBMError as error:
    if 'CUDA Tree Learner was not enabled' not in str(error):
        raise
    print('CUDA build is unavailable in this environment. Falling back to CPU training.')
    model = build_regressor('cpu', 1500)
    model.fit(
        X_train,
        y_train,
        eval_set=[(X_valid, y_valid)],
        eval_metric='rmse',
        categorical_feature=categorical_columns,
        callbacks=[lgb.early_stopping(50, verbose=True)],
    )
    trained_device = 'cpu'

category_levels = {column: combined_df[column].cat.categories for column in categorical_columns}
validation_history_df = train_df.loc[train_df['day'] == 48].copy()
valid_target_df = train_df.loc[valid_mask].copy()
valid_prediction_series = recursive_predict(
    model,
    validation_history_df,
    valid_target_df,
    feature_columns,
    categorical_columns,
    category_levels,
)
valid_predictions = valid_prediction_series.reindex(valid_target_df['Index'].astype(int).tolist()).to_numpy(dtype=float)
valid_r2 = r2_score(y_valid, valid_predictions)
print(f'Validation R2: {valid_r2:.6f}')
print(f'Validation Score: {max(0.0, 100 * valid_r2):.4f}')
print(f'Training device used: {trained_device}')

best_n_estimators = model.best_iteration_ or 1500
full_train_df = train_df.loc[train_df['demand'].notna()].copy()
X_full_train = full_train_df[feature_columns].copy()
y_full_train = full_train_df['demand'].astype(float)

final_model = build_regressor(trained_device, best_n_estimators)
final_model.fit(
    X_full_train,
    y_full_train,
    categorical_feature=categorical_columns,
)

test_history_df = train_df.copy()
test_prediction_series = recursive_predict(
    final_model,
    test_history_df,
    test_df.copy(),
    feature_columns,
    categorical_columns,
    category_levels,
)
test_predictions = test_prediction_series.reindex(test_df['Index'].astype(int).tolist()).to_numpy(dtype=float)
submission = pd.DataFrame({
    'Index': test_df['Index'].astype(int),
    'demand': test_predictions,
})
submission.to_csv(SUBMISSION_PATH, index=False)
submission.head()

CUDA build is unavailable in this environment. Falling back to CPU training.
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[111]	valid_0's rmse: 0.0296812	valid_0's l2: 0.000880972
Validation R2: 0.915908
Validation Score: 91.5908
Training device used: cpu


,Index,demand
12,887,0.024478
13,4491,0.025401
14,21800,0.021762
15,40211,0.019998
17,2,0.027892
